# parqit, part 1: working with data that stay on disk

**Reading, writing, collecting, merging and appending with lazy tables**

parqit lets you work on Parquet files with the Stata commands you already know
(`keep`, `generate`, `collapse`, `merge`, `append`, ...) while the data **stay on disk**.
The commands are carried out by a database engine built into the plugin (DuckDB), *out
of core*: a file can be much larger than your computer's memory, because only the result
you ask for is ever loaded into Stata.

This notebook builds the idea step by step on a small, familiar dataset.
Part 2 (`parqit_2_descriptive_statistics.ipynb`) covers descriptive statistics.

### The one idea: a view is a plan, not data

| Move | Command | What happens |
|---|---|---|
| **Open** | `parqit use using file.parquet` | A *lazy view* opens. parqit reads the file's column names and types, **not its rows**. |
| **Shape** | `parqit keep if ...`, `parqit gen ...`, `parqit merge ...` | Each command adds one step to a plan. **The result is not computed yet.** |
| **Look** | `parqit describe`, `parqit count`, `parqit summarize` | The engine computes the answer on the file; only the printed output comes back. |
| **Land** | `parqit collect` or `parqit save` | The whole plan runs as one query. The result goes into Stata's memory (`collect`) or into a new Parquet file (`save`). |

```
 file.parquet ──►  lazy view = a plan  ──►  parqit collect ──►  Stata's memory
   on disk         keep · gen · merge  ──►  parqit save    ──►  result.parquet
```

**Stata only needs room for the result you collect, not for the file you read.**
To make this visible, the notebook keeps printing how many observations are in Stata's
memory.

### Before you start

* You need Stata 16 or newer and parqit **0.2.1** or later (installed in the first code
  cell below).
* To run the notebook you need Jupyter with the **nbstata** kernel:
  `pip install nbstata`, then `python -m nbstata.install` (see the nbstata documentation
  if your Stata installation is not detected). Choose the kernel *Stata (nbstata)* and
  run the cells in order.
* You can also simply read it: the outputs below were produced with parqit 0.2.1.
* In the Stata windows interface, `parqit menu` adds *User > parqit* dialogs that write
  these same commands.

**The data.** `nlsw88` is an extract of the U.S. National Longitudinal Survey of Young
Women in 1988: 2,246 women, with their hourly wage, hours, industry, union membership
and more. It ships with every copy of Stata (`sysuse nlsw88`) and is small on purpose,
so that every result is easy to check. Section 8 builds a copy 1,000 times larger.

## 0. Setup

The first time, install parqit from the project's GitHub releases by removing the
`*` in front of `net install` (`latest` is version 0.2.1 at the time of writing; to
install exactly that version, replace `latest/download` with `download/v0.2.1`).
Restart Stata (here: the kernel) after installing or updating parqit.

`parqit version` must report 0.2.1 or later. The global `NB` names the scratch folder
where every file of this notebook is written, under Stata's temporary directory.

In [1]:
* First time only: install parqit 0.2.1 from GitHub
* net install parqit, from("https://github.com/reisportela/parqit/releases/latest/download") replace
set linesize 100
parqit version
global NB "`c(tmpdir)'/parqit_notebooks"
capture mkdir "$NB"
display as text "Files for this notebook go to: " as result "$NB"

parqit version 0.2.1  (engine: DuckDB v1.5.3, Stata Plugin Interface 3.0, parallelism: DuckDB)
CPUs available: 48; engine threads: 48; fill workers: auto; stream buffer (MB): auto
Files for this notebook go to: /tmp/parqit_notebooks


## 1. Write a Parquet file from Stata

Start in plain Stata: load the survey and write it to Parquet. The `data` option means
*write the dataset in memory*; without it, `parqit save` writes the current *view*, as
we will see later. Variable labels, value labels, notes and display formats are stored
inside the file, which remains an ordinary Parquet file that Python, R or Spark can read.

In [2]:
sysuse nlsw88, clear
describe, short
parqit save "$NB/nlsw88.parquet", replace data compression(zstd)

(NLSW, 1988 extract)

Contains data from /usr/local/stata/ado/base/n/nlsw88.dta
 Observations:         2,246                  NLSW, 1988 extract
    Variables:            17                  1 May 2024 22:52
Sorted by: idcode
(2246 obs, 17 vars written to /tmp/parqit_notebooks/nlsw88.parquet)


## 2. Look at a file before reading it

`parqit describe` with a file name reads only the file's *footer*: a small block of
metadata at the end of every Parquet file with the number of rows, the columns and
their types, so it is just as quick for a file with billions of rows. We even clear
Stata's memory first.

In [3]:
clear
parqit describe "$NB/nlsw88.parquet"


  /tmp/parqit_notebooks/nlsw88.parquet
  rows:                2,246   columns: 17   row groups: 1   files: 1   parqit metadata: yes

  variable                         parquet type       stata type format
  ------------------------------------------------------------------------
  idcode                           SMALLINT           int        %8.0g
  age                              TINYINT            byte       %8.0g
  race                             TINYINT            byte       %8.0g
  married                          TINYINT            byte       %8.0g
  never_married                    TINYINT            byte       %16.0g
  grade                            TINYINT            byte       %8.0g
  collgrad                         TINYINT            byte       %16.0g
  south                            TINYINT            byte       %9.0g
  smsa                             TINYINT            byte       %9.0g
  c_city                           TINYINT            byte       %16.0g
  indu

## 3. Read a file into memory, when it fits

For a small file, `parqit use ..., clear` works like `use`: the data go straight into
memory. List the variables you need and only those columns are read from disk. The
labels saved in Section 1 are back.

In [4]:
parqit use idcode age race married wage industry using "$NB/nlsw88.parquet", clear
describe
list in 1/5

(6 vars, 2246 obs read from /tmp/parqit_notebooks/nlsw88.parquet)

Contains data
 Observations:         2,246                  NLSW, 1988 extract
    Variables:             6                  
                                              (_dta has notes)
----------------------------------------------------------------------------------------------------
Variable      Storage   Display    Value
    name         type    format    label      Variable label
----------------------------------------------------------------------------------------------------
idcode          int     %8.0g                 NLS ID
age             byte    %8.0g                 Age in current year
race            byte    %8.0g      racelbl    Race
married         byte    %8.0g      marlbl     Married
wage            float   %9.0g                 Hourly wage
industry        byte    %23.0g     indlbl     Industry
----------------------------------------------------------------------------------------------------
So

## 4. Open a lazy view: the file stays on disk

Now the central idea. Suppose you are busy with other data, here Stata's `auto` data
(74 cars). Without `clear`, `parqit use using` opens a **lazy view** of the file and
leaves your data alone. A view is like a frame that lives on disk.

In [5]:
sysuse auto, clear
parqit use using "$NB/nlsw88.parquet"
display as text "Stata's memory: " as result _N as text " observations (still the 74 cars)"

(1978 automobile data)
(lazy view default opened over /tmp/parqit_notebooks/nlsw88.parquet: 17 columns; schema probed, no r
> ows loaded — use parqit collect or parqit save)
Stata's memory: 74 observations (still the 74 cars)


You can explore the view as if it were loaded. Each command below runs a query on the
file and brings back only what it prints: some rows, the number of rows (2,246) and a
summary. (`parqit head` shows the first rows of every variable; `parqit list` lets you
choose the variables and rows.)

In [6]:
parqit list idcode age race married wage hours industry in 1/5


     +----------------------------------------------------------------------------+
     | idcode   age    race   married       wage   hours                 industry |
     |----------------------------------------------------------------------------|
  1. |      1    37   Black    Single   11.73913      48   Transport/Comm/Utility |
  2. |      2    37   Black    Single   6.400963      40            Manufacturing |
  3. |      3    42   Black    Single   5.016723      40            Manufacturing |
  4. |      4    43   White   Married   9.033813      42    Professional services |
  5. |      6    42   White   Married   8.083731      48            Manufacturing |
     +----------------------------------------------------------------------------+


In [7]:
parqit count
parqit summarize wage hours tenure
display as text "Stata's memory: " as result _N as text " observations"

                  2,246

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
        wage |      2,246    7.766949    5.755523   1.004952   40.74659
       hours |      2,242    37.21811    10.50914          1         80
      tenure |      2,231     5.97785    5.510331          0   25.91667
Stata's memory: 74 observations


> **Key idea.** `list`, `count` and `summarize` were computed on the file. A few lines
> of output came back; the `auto` data in memory were not touched.

## 5. Build a plan with Stata verbs

The verbs you know (`keep`, `drop`, `generate`, `replace`, `egen`, `rename`, `sort`,
`collapse`, `merge`, `append`) work on the view when you put `parqit` in front. They do
**not** produce their result: each one adds a step to the view's plan. parqit checks
names and expressions at once, so a typo fails immediately, but nothing is loaded and
the result is only computed when you land it.

Put `keep` and `keep if` first: the engine then reads fewer columns and rows.

In [8]:
parqit keep if age >= 40 & !missing(wage, hours)
parqit keep idcode age collgrad industry wage hours
parqit gen double weekly_pay = wage * hours
parqit egen double industry_mean = mean(wage), by(industry)
parqit gen double relative_wage = wage / industry_mean
parqit sort idcode
parqit describe
display as text "Stata's memory: " as result _N as text " observations"


  lazy view over /tmp/parqit_notebooks/nlsw88.parquet
  columns: 9   pipeline steps: 5

  variable                         kind     format       label
  ------------------------------------------------------------------------
  idcode                           numeric  %8.0g        NLS ID
  age                              numeric  %8.0g        Age in current year
  collgrad                         numeric  %16.0g       College graduate
  industry                         numeric  %23.0g       Industry
  wage                             numeric  %9.0g        Hourly wage
  hours                            numeric  %8.0g        Usual hours worked
  weekly_pay                       numeric               
  industry_mean                    numeric               
  relative_wage                    numeric               

Stata's memory: 74 observations


The view now has five pipeline steps (`sort` only records the order in which the
result will come out), and Stata's memory still holds the cars.

`parqit show` prints the plan as the single SQL query the engine will run, with one
block per verb. You never have to write or read SQL; it is shown here only to make two
points: **nothing has been computed yet**, and the whole pipeline will run as one query.
The SQL has long lines, so this cell widens Stata's output for a moment.

In [9]:
set linesize 255
parqit show
set linesize 100

-- source: /tmp/parqit_notebooks/nlsw88.parquet
WITH __parqit_s0 AS (
  SELECT "idcode" AS "idcode", "age" AS "age", "race" AS "race", "married" AS "married", "never_married" AS "never_married", "grade" AS "grade", "collgrad" AS "collgrad", "south" AS "so
> uth", "smsa" AS "smsa", "c_city" AS "c_city", "industry" AS "industry", "occupation" AS "occupation", "union" AS "union", CASE WHEN "wage" IS NULL OR NOT isfinite(CAST("wage" AS DOUBLE)) OR abs(CAST("wage" AS DOUBLE)) >= 8.9884656743115795e+307 THEN NUL
> L ELSE "wage" END AS "wage", "hours" AS "hours", CASE WHEN "ttl_exp" IS NULL OR NOT isfinite(CAST("ttl_exp" AS DOUBLE)) OR abs(CAST("ttl_exp" AS DOUBLE)) >= 8.9884656743115795e+307 THEN NULL ELSE "ttl_exp" END AS "ttl_exp", CASE WHEN "tenure" IS NULL OR
>  NOT isfinite(CAST("tenure" AS DOUBLE)) OR abs(CAST("tenure" AS DOUBLE)) >= 8.9884656743115795e+307 THEN NULL ELSE "tenure" END AS "tenure" FROM read_parquet(['/tmp/parqit_notebooks/nlsw88.parquet'])
),
-- keep if   age >= 40 & !m

Stripped of details, the query has one block per verb, in the order you typed them:

```
WITH __parqit_s0 AS (... read nlsw88.parquet ...),
-- keep if   age >= 40 & !missing(wage, hours)
__parqit_s1 AS (...),
-- keep 6 variables
__parqit_s2 AS (...),
-- gen weekly_pay = wage * hours
__parqit_s3 AS (...),
-- egen industry_mean = mean(...), by(...)
__parqit_s4 AS (...),
-- gen relative_wage = wage / industry_mean
__parqit_s5 AS (...)
SELECT * FROM __parqit_s5 ORDER BY "idcode" NULLS LAST
```

The long `CASE WHEN ... isfinite(...)` expressions are parqit's guards that make numbers
behave as in Stata (for example, an infinite value becomes missing).

## 6. Land the result: `collect` into memory, or `save` to disk

`parqit collect` runs the plan and puts the result in Stata's memory. It is
**atomic**: your data are replaced only once the new data are complete. The view stays
open, so you can keep working with it (collecting again re-runs the plan).

In [10]:
parqit collect, clear
describe, short
list idcode age wage industry_mean relative_wage in 1/5, abbreviate(14)

(9 vars, 995 obs collected; view default remains open)

Contains data
 Observations:           995                  NLSW, 1988 extract
    Variables:             9                  
Sorted by: idcode

     +---------------------------------------------------------+
     | idcode   age       wage   industry_mean   relative_wage |
     |---------------------------------------------------------|
  1. |      3    42   5.016723       7.4364221        .6746151 |
  2. |      4    43   9.033813       7.8392333       1.1523847 |
  3. |      6    42   8.083731       7.4364221       1.0870457 |
  4. |     12    40   17.20612       7.8392333       2.1948723 |
  5. |     13    40   13.08374       7.8392333        1.669007 |
     +---------------------------------------------------------+


When the result is too large for memory, or you simply want a clean file for later,
`parqit save` runs the same plan and writes a new Parquet file. The result never passes
through Stata's memory: the cars stay where they are.

In [11]:
sysuse auto, clear
parqit save "$NB/nlsw88_age40plus.parquet", replace compression(zstd)
display as text "Stata's memory: " as result _N as text " observations"
parqit describe "$NB/nlsw88_age40plus.parquet"
parqit close

(1978 automobile data)
(materialising view default — the dataset in memory is untouched; use the data option to export memo
> ry instead)
(995 obs, 9 vars written to /tmp/parqit_notebooks/nlsw88_age40plus.parquet)
Stata's memory: 74 observations

  /tmp/parqit_notebooks/nlsw88_age40plus.parquet
  rows:                  995   columns: 9   row groups: 1   files: 1   parqit metadata: yes

  variable                         parquet type       stata type format
  ------------------------------------------------------------------------
  idcode                           SMALLINT           int        %8.0g
  age                              TINYINT            byte       %8.0g
  collgrad                         TINYINT            byte       %16.0g
  industry                         TINYINT            byte       %23.0g
  wage                             FLOAT              float      %9.0g
  hours                            TINYINT            byte       %8.0g
  weekly_pay                       D

## 7. Aggregate on disk, collect only the answer

A typical research workflow turns a large file of individual records into a small
table: averages by industry, firm, region or year. `collapse` runs in the engine and
only the aggregated rows reach Stata. `gsort -mean_wage` sorts the table from the
highest mean wage down.

In [12]:
parqit use using "$NB/nlsw88.parquet"
parqit collapse (mean) mean_wage=wage (p50) median_wage=wage (count) n=wage, by(industry)
parqit gsort -mean_wage
parqit collect, clear
list, noobs abbreviate(12)

(lazy view default opened over /tmp/parqit_notebooks/nlsw88.parquet: 17 columns; schema probed, no r
> ows loaded — use parqit collect or parqit save)
(4 vars, 13 obs collected; view default remains open)

  +---------------------------------------------------------+
  |                industry   mean_wage   median_wage     n |
  |---------------------------------------------------------|
  |                  Mining    15.34959      8.091784     4 |
  |  Transport/Comm/Utility    11.44335       10.1248    90 |
  | Finance/Ins/Real estate    9.843174      7.053139   192 |
  |   Public administration    9.148407      8.397742   176 |
  |   Professional services    7.871186       6.69887   824 |
  |---------------------------------------------------------|
  |            Construction    7.564934      6.688963    29 |
  |     Business/Repair svc     7.51579      5.334139    86 |
  |           Manufacturing    7.501578      6.191625   367 |
  |   Entertainment/Rec svc    6.724409      4.227

The row whose industry is a dot (`.`) holds the 14 women with no industry recorded:
like Stata's own `collapse`, parqit keeps them as a group.

## 8. The same code on a file 1,000 times larger

Nothing above depends on the size of the file, and that is the point of parqit. To see
it, we build a file with **2,246,000 rows** that never enters Stata. `parqit sql` opens
a view on any DuckDB query; this one repeats each of the 2,246 rows 1,000 times (the new
variable `copy` numbers the repetitions), and `parqit save` writes the result to disk.

This is a synthetic enlargement: every woman appears 1,000 times, so averages are those
of the original file. A raw SQL result carries no Stata labels, so `industry` shows its
numeric codes.

In [13]:
sysuse auto, clear
parqit sql "select n.*, r.range + 1 as copy from read_parquet('$NB/nlsw88.parquet') n, range(1000) r"
parqit save "$NB/nlsw88_x1000.parquet", replace compression(zstd)
parqit close
parqit describe "$NB/nlsw88_x1000.parquet"
display as text "Stata's memory: " as result _N as text " observations"

(1978 automobile data)
(view default opened over the SQL result: 18 columns)
(materialising view default — the dataset in memory is untouched; use the data option to export memo
> ry instead)
(2246000 obs, 18 vars written to /tmp/parqit_notebooks/nlsw88_x1000.parquet)
(current view closed)

  /tmp/parqit_notebooks/nlsw88_x1000.parquet
  rows:            2,246,000   columns: 18   row groups: 19   files: 1   parqit metadata: yes

  variable                         parquet type       stata type format
  ------------------------------------------------------------------------
  idcode                           SMALLINT           long       
  age                              TINYINT            int        
  race                             TINYINT            int        
  married                          TINYINT            int        
  never_married                    TINYINT            int        
  grade                            TINYINT            int        
  collgrad               

Now the same commands as before. Stata's memory keeps holding the 74 cars while the
engine reads the 2,246,000 rows.

In [14]:
parqit use using "$NB/nlsw88_x1000.parquet"
parqit count
parqit summarize wage hours
display as text "Stata's memory: " as result _N as text " observations"

(lazy view default opened over /tmp/parqit_notebooks/nlsw88_x1000.parquet: 18 columns; schema probed
> , no rows loaded — use parqit collect or parqit save)
              2,246,000

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
        wage |    2246000    7.766949    5.754243   1.004952   40.74659
       hours |    2242000    37.21811    10.50679          1         80
Stata's memory: 74 observations


In [15]:
parqit collapse (mean) mean_wage=wage (count) n=wage, by(industry)
parqit gsort -mean_wage
parqit collect, clear
format mean_wage %9.0g
list, noobs abbreviate(12)

(3 vars, 13 obs collected; view default remains open)

  +-------------------------------+
  | industry   mean_wage        n |
  |-------------------------------|
  |        2    15.34959     4000 |
  |        5    11.44335    90000 |
  |        7    9.843174   192000 |
  |       12    9.148407   176000 |
  |       11    7.871186   824000 |
  |-------------------------------|
  |        3    7.564934    29000 |
  |        8     7.51579    86000 |
  |        4    7.501578   367000 |
  |       10    6.724409    17000 |
  |        6    6.125897   333000 |
  |-------------------------------|
  |        1    5.621121    17000 |
  |        .     5.13411    14000 |
  |        9    4.401093    97000 |
  +-------------------------------+


(The file built with SQL carries no display formats, so `format` gives `mean_wage` the
one it had in Section 7.) The mean wages match Section 7 and every count is 1,000 times
larger, yet Stata only ever held a 13-row table.

To go further, change `range(1000)` to `range(10000)` (22.5 million rows) and rerun:
the commands stay the same, the engine does more work, and Stata still receives only
the 13-row result.

> **Key idea.** With parqit, the memory Stata needs is set by the result you collect,
> not by the size of the file you work on.

## 9. Merge: bring in information from another file

A small lookup table gives each of the 12 industries a broad sector. We type it into
memory and write it to Parquet, as you would with any lookup or crosswalk file.

In [16]:
clear
input byte industry str9 sector
 1 "Primary"
 2 "Primary"
 3 "Secondary"
 4 "Secondary"
 5 "Services"
 6 "Services"
 7 "Services"
 8 "Services"
 9 "Services"
10 "Services"
11 "Services"
12 "Services"
end
label variable sector "Broad sector"
parqit save "$NB/sectors.parquet", replace data compression(zstd)


(12 obs, 2 vars written to /tmp/parqit_notebooks/sectors.parquet)


`parqit merge` joins the view with a file, lazily and on disk, with the syntax of
`merge` (`1:1`, `m:1`, `1:m` and the options `keep()`, `keepusing()`, `generate()` and
`nogenerate`). Like `merge`, it checks at once that the key identifies the rows the
merge type requires (here, each industry appears only once in the lookup file). It
creates the familiar `_merge` variable:

In [17]:
parqit use using "$NB/nlsw88.parquet"
parqit merge m:1 industry using "$NB/sectors.parquet", keepusing(sector)
parqit tabulate _merge

(lazy view default opened over /tmp/parqit_notebooks/nlsw88.parquet: 17 columns; schema probed, no r
> ows loaded — use parqit collect or parqit save)

 Matching result from merge |      Freq.     Percent        Cum.
----------------------------+-----------------------------------
            Master only (1) |         14        0.62        0.62
                Matched (3) |      2,232       99.38      100.00
----------------------------+-----------------------------------
                      Total |      2,246      100.00


The 14 unmatched women (`_merge == 1`, *master only*) are those with no industry: a
missing key has no match in the lookup, exactly as native `merge` would report. The plan
continues: keep the matched rows and compute mean wages by sector.

In [18]:
parqit keep if _merge == 3
parqit drop _merge
parqit collapse (mean) mean_wage=wage (count) n=wage, by(sector)
parqit collect, clear
list, noobs abbreviate(10)

(3 vars, 3 obs collected; view default remains open)

  +------------------------------+
  |    sector   mean_wage      n |
  |------------------------------|
  |   Primary    7.474163     21 |
  | Secondary    7.506218    396 |
  |  Services    7.847532   1815 |
  +------------------------------+


**Data already in memory?** Then use `parqit mergein`: it runs Stata's own `merge`
between the data in memory and a Parquet file, reading only the columns it needs from
disk. It takes the native options (`keep()`, `keepusing()`, `assert()`, `update`, ...).

In [19]:
sysuse nlsw88, clear
parqit mergein m:1 industry using "$NB/sectors.parquet", keepusing(sector)
tabulate sector, missing

(NLSW, 1988 extract)

    Result                      Number of obs
    -----------------------------------------
    Not matched                            14
        from master                        14  (_merge==1)
        from using                          0  (_merge==2)

    Matched                             2,232  (_merge==3)
    -----------------------------------------

      Broad |
     sector |      Freq.     Percent        Cum.
------------+-----------------------------------
            |         14        0.62        0.62
    Primary |         21        0.93        1.56
  Secondary |        396       17.63       19.19
   Services |      1,815       80.81      100.00
------------+-----------------------------------
      Total |      2,246      100.00


## 10. Append: stack files

First split the survey into two files, women living in the South and elsewhere. Each
file is written straight from a view to disk:

In [20]:
parqit use using "$NB/nlsw88.parquet"
parqit keep if south == 1
parqit save "$NB/region_south.parquet", replace compression(zstd)
parqit use using "$NB/nlsw88.parquet"
parqit keep if south == 0
parqit save "$NB/region_other.parquet", replace compression(zstd)
parqit close

(lazy view default opened over /tmp/parqit_notebooks/nlsw88.parquet: 17 columns; schema probed, no r
> ows loaded — use parqit collect or parqit save)
(materialising view default — the dataset in memory is untouched; use the data option to export memo
> ry instead)
(942 obs, 17 vars written to /tmp/parqit_notebooks/region_south.parquet)
(lazy view default opened over /tmp/parqit_notebooks/nlsw88.parquet: 17 columns; schema probed, no r
> ows loaded — use parqit collect or parqit save)
(materialising view default — the dataset in memory is untouched; use the data option to export memo
> ry instead)
(1304 obs, 17 vars written to /tmp/parqit_notebooks/region_other.parquet)
(current view closed)


`parqit append` stacks files lazily, matching columns by name. `generate()` records
where each row came from (0 = the view, 1 = the first appended file):

In [21]:
parqit use using "$NB/region_south.parquet"
parqit append using "$NB/region_other.parquet", generate(source)
parqit tabulate source
parqit close

(lazy view default opened over /tmp/parqit_notebooks/region_south.parquet: 17 columns; schema probed
> , no rows loaded — use parqit collect or parqit save)

 Source of the observation (0 = master) |      Freq.     Percent        Cum.
----------------------------------------+-----------------------------------
                                      0 |        942       41.94       41.94
                                      1 |      1,304       58.06      100.00
----------------------------------------+-----------------------------------
                                  Total |      2,246      100.00
(current view closed)


**Many files, one table.** Data often arrive as many files (one per month, region,
...). A wildcard opens every matching file as a single view, so no `append` is needed.
All 2,246 women are back:

In [22]:
parqit use using "$NB/region_*.parquet"
parqit count
parqit close

(lazy view default opened over /tmp/parqit_notebooks/region_*.parquet: 17 columns; schema probed, no
>  rows loaded — use parqit collect or parqit save)
                  2,246
(current view closed)


**Data already in memory?** `parqit appendin` is the counterpart of `mergein`: Stata's
own `append` of a Parquet file to the data in memory. The `label ... already defined`
notes come from Stata's `append`: both files carry the same value labels.

In [23]:
parqit use "$NB/region_south.parquet", clear
parqit appendin using "$NB/region_other.parquet"
count

(17 vars, 942 obs read from /tmp/parqit_notebooks/region_south.parquet)
(label unionlbl already defined)
(label southlbl already defined)
(label smsalbl already defined)
(label racelbl already defined)
(label occlbl already defined)
(label nev_mar already defined)
(label marlbl already defined)
(label indlbl already defined)
(label gradlbl already defined)
(label ccitylbl already defined)
  2,246


## 11. Folders of files: partitioned datasets

Large datasets are often stored as a folder with one subfolder per value of a key, for
example `year=2020/`, `year=2021/`: a *Hive-partitioned* dataset. `partition_by()`
writes one, and `parqit use` reads the whole folder back as one table.

In [24]:
parqit use using "$NB/nlsw88.parquet"
parqit save "$NB/nlsw88_by_race", replace partition_by(race) compression(zstd)
parqit close
local folders : dir "$NB/nlsw88_by_race" dirs "*"
local folders : list sort folders
display as text "Subfolders: " as result `"`folders'"'

(lazy view default opened over /tmp/parqit_notebooks/nlsw88.parquet: 17 columns; schema probed, no r
> ows loaded — use parqit collect or parqit save)
(materialising view default — the dataset in memory is untouched; use the data option to export memo
> ry instead)
(2246 obs, 17 vars written to /tmp/parqit_notebooks/nlsw88_by_race)
(current view closed)
Subfolders: "race=1" "race=2" "race=3"


The folder reads back as one table of 2,246 rows. A filter on the partition key lets
the engine skip the subfolders that cannot match: after `keep if race == 2`, only the
`race=2/` subfolder is read to count the 583 Black women.

In [25]:
parqit use using "$NB/nlsw88_by_race"
parqit count
parqit keep if race == 2
parqit count
parqit close

(lazy view default opened over /tmp/parqit_notebooks/nlsw88_by_race: 17 columns; schema probed, no r
> ows loaded — use parqit collect or parqit save)
                  2,246
                    583
(current view closed)


## 12. Summary

| You want to... | Command |
|---|---|
| See a file's rows, columns and types | `parqit describe file.parquet` |
| Read a file into memory | `parqit use [varlist] using file.parquet, clear` |
| Open a lazy view (nothing loaded) | `parqit use using file.parquet` (also `*.parquet` or a partitioned folder) |
| Peek at the view | `parqit head`, `parqit list`, `parqit count`, `parqit describe` |
| Transform lazily | `parqit keep`, `drop`, `gen`, `replace`, `egen`, `rename`, `sort`, `gsort`, `collapse` |
| Combine lazily | `parqit merge`, `parqit append` |
| Combine with the data in memory | `parqit mergein`, `parqit appendin` |
| See the plan | `parqit show` |
| Land the result in memory | `parqit collect, clear` |
| Land the result on disk | `parqit save result.parquet, replace` |
| Write the data in memory | `parqit save file.parquet, replace data` |
| Close views | `parqit close`, `parqit close _all` |

**The habit:** explore first, build one lazy plan, and land the result last.

Next: `parqit_2_descriptive_statistics.ipynb` computes descriptive statistics on disk.
The full manual is `help parqit`, and two runnable do-file courses ship with the
package, `parqit_basics.do` and `parqit_tour.do` (`net get parqit`, from the address
used to install it, copies them into the current folder).

In [26]:
parqit close _all

(views closed)
